# Model Training + Submission Pipeline

This notebook implements:
- MLflow tracking for scout/full runs,
- LOGO/LORO-safe grouped evaluation,
- frozen feature sets (A/B),
- target-specific model strategy,
- frozen manifest (no submission-time auto-picking),
- 3-shot submission ladder (A safe, B EC-aggressive + DRP-safe, C blend).


In [ ]:
import os
import sys
import json
import time
import hashlib
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from xgboost import XGBRegressor

# -------------------------------
# Environment + MLflow
# -------------------------------
sys.path.append(os.path.abspath('..'))
ENV = 'local'  # switch to 'snowflake' when needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')
print('MLflow URI:', config.MLFLOW_URI)

# -------------------------------
# Global config
# -------------------------------
TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
SPLIT_STRATEGY = 'LORO'
GROUP_DEFINITION_VERSION = 'region_v1'
PIPELINE_VERSION = 'deadline_v1_no_stack'
PREPROCESS_VERSION = 'median_var0_scaler'
ARTIFACT_DIR = '../models/final_deadline'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

def hash_str(s: str) -> str:
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]

def hash_list(values) -> str:
    return hash_str('||'.join(map(str, values)))

def compute_group_values_hash(groups: pd.Series) -> str:
    return hash_list(groups.fillna('NA').astype(str).tolist())

def compute_feature_set_hash(features: list) -> str:
    return hash_list(sorted(features))

def target_key(t: str) -> str:
    return t.replace(' ', '')

def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out

def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')
    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')
    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')
    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')
    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')
    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')

def log_common_run_context(target, cand_id, model_name, feature_set_name, features_used, stage, groups_series):
    mlflow.log_param('stage', stage)
    mlflow.log_param('target', target)
    mlflow.log_param('candidate_id', cand_id)
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('feature_set_name', feature_set_name)
    mlflow.log_param('split_strategy', SPLIT_STRATEGY)
    mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
    mlflow.log_param('group_values_hash', compute_group_values_hash(groups_series))
    mlflow.log_param('feature_set_hash', compute_feature_set_hash(features_used))
    mlflow.log_param('pipeline_version', PIPELINE_VERSION)
    mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
    mlflow.log_param('n_features_used', len(features_used))

# -------------------------------
# Load data
# -------------------------------
df = config.load_data()
if 'Region' not in df.columns:
    raise RuntimeError('Region column is required for LORO but was not found.')
if df['Region'].nunique() < 2:
    raise RuntimeError('Need at least 2 unique regions for grouped CV.')

# -------------------------------
# Feature engineering
# -------------------------------
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    df_eng = data.copy()
    eps = 1e-5

    if {'worldpop_mean_1km', 'basin_upstream_area_km2'}.issubset(df_eng.columns):
        df_eng['pop_density_upstream'] = df_eng['worldpop_mean_1km'] / (df_eng['basin_upstream_area_km2'] + eps)

    if {'river_avg_discharge_cms', 'basin_upstream_area_km2'}.issubset(df_eng.columns):
        df_eng['specific_discharge'] = df_eng['river_avg_discharge_cms'] / (df_eng['basin_upstream_area_km2'] + eps)

    if {'pet', 'total_precipitation'}.issubset(df_eng.columns):
        df_eng['aridity_index'] = df_eng['pet'] / (df_eng['total_precipitation'] + eps)

    if {'average_wind_speed', 'basin_slope_deg'}.issubset(df_eng.columns):
        df_eng['wind_exposure'] = df_eng['average_wind_speed'] * (1.0 + df_eng['basin_slope_deg'])

    if {'NDMI', 'pet'}.issubset(df_eng.columns):
        df_eng['moisture_heat_ratio'] = df_eng['NDMI'] / (df_eng['pet'] + eps)

    num_cols = df_eng.select_dtypes(include=[np.number]).columns
    df_eng[num_cols] = df_eng[num_cols].replace([np.inf, -np.inf], np.nan)
    return df_eng

df = engineer_features(df)

# -------------------------------
# Frozen feature sets
# -------------------------------
BASE_FEATURES = [
    'nir','green','swir16','swir22','NDMI','MNDWI','pet',
    'elevation_meters','total_precipitation','average_wind_speed',
    'soil_phh2o_mean_0_5cm','soil_clay_mean_0_5cm','soil_sand_mean_0_5cm',
    'soil_silt_mean_0_5cm','soil_organic_carbon_mean_0_5cm','soil_cec_mean_0_5cm',
    'sanlc2022_impact_1km','sanlc2020_impact_1km','sanlc_change_2020_2022',
    'worldpop_mean_1km','basin_upstream_area_km2','basin_population',
    'basin_agriculture_pct','basin_slope_deg','river_avg_discharge_cms',
    'river_order','river_width_m','river_upstream_area_km2'
]

EMPIRICAL_V2 = [
    'aridity_index',
    'wind_exposure',
    'moisture_heat_ratio',
    'pop_density_upstream',
    'specific_discharge'
]

FEATURE_SETS = {
    'A': BASE_FEATURES,
    'B': BASE_FEATURES + EMPIRICAL_V2,
}

def features_for_set(df_local: pd.DataFrame, fs_name: str):
    requested = FEATURE_SETS[fs_name]
    present = [f for f in requested if f in df_local.columns]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        print(f'[WARN] Missing in {fs_name}: {missing}')
    if len(present) == 0:
        raise RuntimeError(f'No usable features for set {fs_name}')
    return present

def get_preprocessor(features_used):
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('var0', VarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler())
    ])
    return ColumnTransformer(
        transformers=[('num', numeric_pipe, features_used)],
        remainder='drop'
    )

def log_wrap(model):
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

def build_candidates():
    cands = []

    # TA
    cands += [
        {'cand_id': 'TA_XGB_A', 'target': 'Total Alkalinity', 'feature_set': 'A', 'model_name': 'XGB',
         'estimator': log_wrap(XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.7, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1))},
        {'cand_id': 'TA_XGB_B', 'target': 'Total Alkalinity', 'feature_set': 'B', 'model_name': 'XGB',
         'estimator': log_wrap(XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.7, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1))}
    ]

    # EC
    cands += [
        {'cand_id': 'EC_LASSO_B', 'target': 'Electrical Conductance', 'feature_set': 'B', 'model_name': 'Lasso', 'estimator': log_wrap(Lasso(alpha=0.001))},
        {'cand_id': 'EC_ELASTIC_B', 'target': 'Electrical Conductance', 'feature_set': 'B', 'model_name': 'ElasticNet', 'estimator': log_wrap(ElasticNet(alpha=0.001, l1_ratio=0.7, random_state=42))},
        {'cand_id': 'EC_XGB_B', 'target': 'Electrical Conductance', 'feature_set': 'B', 'model_name': 'XGB',
         'estimator': log_wrap(XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.7, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1))}
    ]

    # DRP
    cands += [
        {'cand_id': 'DRP_RIDGE_B', 'target': 'Dissolved Reactive Phosphorus', 'feature_set': 'B', 'model_name': 'Ridge', 'estimator': log_wrap(Ridge(alpha=10.0))},
        {'cand_id': 'DRP_LASSO_B', 'target': 'Dissolved Reactive Phosphorus', 'feature_set': 'B', 'model_name': 'Lasso', 'estimator': log_wrap(Lasso(alpha=0.001))},
        {'cand_id': 'DRP_XGB_B', 'target': 'Dissolved Reactive Phosphorus', 'feature_set': 'B', 'model_name': 'XGB',
         'estimator': log_wrap(XGBRegressor(n_estimators=220, learning_rate=0.03, max_depth=3, subsample=0.7, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=6.0, random_state=42, n_jobs=-1))}
    ]

    return cands

candidates = build_candidates()
print('Candidates:', len(candidates))


In [ ]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None):
    d = df_local.copy()
    if allowed_regions is not None:
        d = d[d['Region'].isin(allowed_regions)].copy()

    X = d[features_used]
    y = d[target].astype(float)
    groups = d['Region'].astype(str)

    logo = LeaveOneGroupOut()
    pipe = Pipeline([
        ('preprocessor', get_preprocessor(features_used)),
        ('model', clone(estimator))
    ])

    pred = cross_val_predict(
        pipe, X, y, groups=groups, cv=logo, n_jobs=-1, verbose=0
    )

    return {
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
        'mae': float(mean_absolute_error(y, pred)),
        'r2': float(r2_score(y, pred)),
        'n_rows': int(len(d)),
        'n_groups': int(groups.nunique())
    }

def fit_full_and_save(df_local, target, estimator, features_used, cand_id):
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{cand_id}__{target_key(target)}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{cand_id}__{target_key(target)}__model.joblib')
    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)
    return preproc_path, model_path

# ----------------------------------------
# Stage 1: Scout grouped CV + MLflow logs
# ----------------------------------------
preferred_scout = ['Eastern_Cape', 'Western_Cape', 'Northern_Bulk']
available_regions = set(df['Region'].astype(str).unique().tolist())
SCOUT_REGIONS = [r for r in preferred_scout if r in available_regions]
if len(SCOUT_REGIONS) < 2:
    SCOUT_REGIONS = None

rows_scout = []
group_values_hash = compute_group_values_hash(df['Region'].astype(str))

for c in candidates:
    feats = features_for_set(df, c['feature_set'])
    run_name = f"SCOUT__{c['cand_id']}__{target_key(c['target'])}"

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()
        m = grouped_oof_eval(
            df_local=df,
            target=c['target'],
            estimator=c['estimator'],
            features_used=feats,
            allowed_regions=SCOUT_REGIONS
        )
        dt = time.time() - t0

        log_common_run_context(
            target=c['target'],
            cand_id=c['cand_id'],
            model_name=c['model_name'],
            feature_set_name=c['feature_set'],
            features_used=feats,
            stage='scout',
            groups_series=df['Region'].astype(str)
        )
        mlflow.log_metric('r2', m['r2'])
        mlflow.log_metric('rmse', m['rmse'])
        mlflow.log_metric('mae', m['mae'])
        mlflow.log_metric('cv_time_sec', dt)
        mlflow.log_metric('n_rows', m['n_rows'])
        mlflow.log_metric('n_groups', m['n_groups'])

        rows_scout.append({
            'run_id': mlflow.active_run().info.run_id,
            'stage': 'scout',
            'cand_id': c['cand_id'],
            'target': c['target'],
            'model_name': c['model_name'],
            'feature_set': c['feature_set'],
            'features_used_json': json.dumps(feats),
            'rmse': m['rmse'],
            'mae': m['mae'],
            'r2': m['r2'],
            'cv_time_sec': dt,
            'split_strategy': SPLIT_STRATEGY,
            'group_definition_version': GROUP_DEFINITION_VERSION,
            'group_values_hash': group_values_hash,
            'feature_set_hash': compute_feature_set_hash(feats),
            'pipeline_version': PIPELINE_VERSION,
            'preprocess_version': PREPROCESS_VERSION,
        })

scout_df = pd.DataFrame(rows_scout)
display(scout_df.sort_values(['target', 'r2'], ascending=[True, False]))

# ----------------------------------------
# Stage 2: Full grouped finalists + save
# ----------------------------------------
finalist_ids = (
    scout_df.sort_values(['target', 'r2'], ascending=[True, False])
            .groupby('target')
            .head(2)['cand_id']
            .tolist()
)
finalist_map = {c['cand_id']: c for c in candidates if c['cand_id'] in finalist_ids}

rows_full = []
for cand_id in finalist_ids:
    c = finalist_map[cand_id]
    feats = features_for_set(df, c['feature_set'])
    run_name = f"FULL__{c['cand_id']}__{target_key(c['target'])}"

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()
        m = grouped_oof_eval(
            df_local=df,
            target=c['target'],
            estimator=c['estimator'],
            features_used=feats,
            allowed_regions=None
        )
        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=c['target'],
            estimator=c['estimator'],
            features_used=feats,
            cand_id=cand_id
        )

        log_common_run_context(
            target=c['target'],
            cand_id=c['cand_id'],
            model_name=c['model_name'],
            feature_set_name=c['feature_set'],
            features_used=feats,
            stage='full',
            groups_series=df['Region'].astype(str)
        )
        mlflow.log_metric('r2', m['r2'])
        mlflow.log_metric('rmse', m['rmse'])
        mlflow.log_metric('mae', m['mae'])
        mlflow.log_metric('cv_time_sec', dt)
        mlflow.log_metric('n_rows', m['n_rows'])
        mlflow.log_metric('n_groups', m['n_groups'])
        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        rows_full.append({
            'run_id': mlflow.active_run().info.run_id,
            'stage': 'full',
            'cand_id': cand_id,
            'target': c['target'],
            'model_name': c['model_name'],
            'feature_set': c['feature_set'],
            'features_used_json': json.dumps(feats),
            'rmse': m['rmse'],
            'mae': m['mae'],
            'r2': m['r2'],
            'cv_time_sec': dt,
            'split_strategy': SPLIT_STRATEGY,
            'group_definition_version': GROUP_DEFINITION_VERSION,
            'group_values_hash': group_values_hash,
            'feature_set_hash': compute_feature_set_hash(feats),
            'pipeline_version': PIPELINE_VERSION,
            'preprocess_version': PREPROCESS_VERSION,
            'preproc_path': preproc_path,
            'model_path': model_path
        })

full_df = pd.DataFrame(rows_full)
display(full_df.sort_values(['target', 'r2'], ascending=[True, False]))


In [ ]:
# ----------------------------------------
# Freeze manifest A (safe anchor)
# ----------------------------------------
GATE_R2 = 0.00

drp_linear = full_df[
    (full_df['target'] == 'Dissolved Reactive Phosphorus') &
    (full_df['model_name'].isin(['Ridge', 'Lasso', 'ElasticNet']))
].sort_values('r2', ascending=False)

DRP_SAFE_R2 = float(drp_linear.iloc[0]['r2']) if len(drp_linear) else 0.00
manifest_A = {}

for t in TARGET_COLS:
    tdf = full_df[full_df['target'] == t].sort_values('r2', ascending=False).copy()
    tdf = tdf[tdf['r2'] >= GATE_R2]

    if t == 'Dissolved Reactive Phosphorus':
        tdf = tdf[tdf['r2'] >= DRP_SAFE_R2]

    if tdf.empty:
        fallback = full_df[
            (full_df['target'] == t) &
            (full_df['model_name'].isin(['Ridge', 'Lasso', 'ElasticNet']))
        ].sort_values('r2', ascending=False)
        manifest_A[t] = None if fallback.empty else fallback.iloc[0].to_dict()
    else:
        manifest_A[t] = tdf.iloc[0].to_dict()

print('DRP_SAFE_R2:', DRP_SAFE_R2)
print('\n=== FROZEN MANIFEST A ===')
print(json.dumps({
    k: (None if v is None else {
        'run_id': v['run_id'],
        'cand_id': v['cand_id'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'r2': float(v['r2'])
    })
    for k, v in manifest_A.items()
}, indent=2))

# ----------------------------------------
# Build Shot A, Shot B, Shot C
# ----------------------------------------
def predict_from_manifest_entry(entry, df_val_local):
    feats = json.loads(entry['features_used_json'])
    pre = joblib.load(entry['preproc_path'])
    mdl = joblib.load(entry['model_path'])
    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    return np.clip(pred, 0, None)

df_val = pd.read_parquet('../data/interim/master_test.parquet')
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

# Shot A
shotA = tpl.copy()
for t in TARGET_COLS:
    e = manifest_A[t]
    if e is None:
        raise RuntimeError(f'No manifest entry for target {t}')
    shotA[t] = predict_from_manifest_entry(e, df_val)
assert_submission_integrity(shotA, tpl, TARGET_COLS)

# Shot B
shotB = shotA.copy()

ec_A_r2 = float(manifest_A['Electrical Conductance']['r2'])
ec_pool = full_df[
    (full_df['target'] == 'Electrical Conductance') &
    (full_df['feature_set'] == 'B') &
    (full_df['model_name'].isin(['Lasso', 'ElasticNet', 'XGB']))
].sort_values('r2', ascending=False)

if len(ec_pool) and float(ec_pool.iloc[0]['r2']) > ec_A_r2:
    best_ec = ec_pool.iloc[0].to_dict()
    shotB['Electrical Conductance'] = predict_from_manifest_entry(best_ec, df_val)
    print('EC in Shot B replaced by challenger:', best_ec['cand_id'])
else:
    print('EC in Shot B kept from Shot A (no better challenger).')

drp_pool = full_df[
    (full_df['target'] == 'Dissolved Reactive Phosphorus') &
    (full_df['feature_set'] == 'B') &
    (full_df['model_name'].isin(['Ridge', 'Lasso', 'ElasticNet']))
].sort_values('r2', ascending=False)

if len(drp_pool):
    best_drp = drp_pool.iloc[0].to_dict()
    drp_pred = predict_from_manifest_entry(best_drp, df_val)
    drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())
    drp_pred = 0.6 * drp_pred + 0.4 * drp_train_median
    shotB['Dissolved Reactive Phosphorus'] = np.clip(drp_pred, 0, None)
    print('DRP in Shot B set to safe-linear+shrink:', best_drp['cand_id'])
else:
    print('No DRP safe challenger found; keep Shot A DRP.')

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# Shot C blend hedge
shotC = tpl.copy()
shotC['Total Alkalinity'] = np.clip(0.70 * shotA['Total Alkalinity'] + 0.30 * shotB['Total Alkalinity'], 0, None)
shotC['Electrical Conductance'] = np.clip(0.40 * shotA['Electrical Conductance'] + 0.60 * shotB['Electrical Conductance'], 0, None)
shotC['Dissolved Reactive Phosphorus'] = np.clip(0.70 * shotA['Dissolved Reactive Phosphorus'] + 0.30 * shotB['Dissolved Reactive Phosphorus'], 0, None)
assert_submission_integrity(shotC, tpl, TARGET_COLS)

os.makedirs('../data/submission', exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_ec_aggr_drp_safe.csv'
pathC = f'../data/submission/submission_{stamp}_C_blend.csv'

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)

print('\nShot A stats')
display(shotA[TARGET_COLS].describe().T[['min', 'mean', 'max']])
print('\nShot B stats')
display(shotB[TARGET_COLS].describe().T[['min', 'mean', 'max']])
print('\nShot C stats')
display(shotC[TARGET_COLS].describe().T[['min', 'mean', 'max']])

tracker = pd.DataFrame([
    {'file': pathA, 'hypothesis': 'Safety anchor: manifest-only, no negative-gated target'},
    {'file': pathB, 'hypothesis': 'EC aggressive (Set B), DRP conservative shrink'},
    {'file': pathC, 'hypothesis': 'Blend hedge between A and B'}
])
display(tracker)

print('\nUpload sequence: A -> B -> C')
print('Stop rule: if A is catastrophically below current best LB, hold prior best and avoid new risky variants.')
